# Week 6 Exercise - Frontier Model Price Prediction with Sampling Hyperparameters

Instead of fine-tuning, this notebook explores how **sampling hyperparameters** (`temperature=0.0`, `top_p=0.1`) can make frontier models more deterministic and improve price prediction accuracy.

Models compared:
- **GPT-4o-mini** (OpenAI)
- **Gemini 2.5 Flash** (Google)
- **Claude 3.5 Haiku** (Anthropic)

In [ ]:
import os
import sys
from dotenv import load_dotenv
from huggingface_hub import login
from litellm import completion

sys.path.insert(0, os.path.abspath("../../"))
from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
login(os.environ['HF_TOKEN'], add_to_git_credential=True)

In [ ]:
# Load dataset

train, val, test = Item.from_hub("ed-donner/items_lite")
print(f"Loaded {len(train):,} training, {len(val):,} validation, {len(test):,} test items")

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
# Quick sanity check
print(test[0].summary)
print(f"Actual price: ${test[0].price:.2f}")

## GPT-4o-mini

In [ ]:
def gpt_4o_mini(item):
    response = completion(
        model="openrouter/openai/gpt-4o-mini",
        messages=messages_for(item),
        temperature=0.0,
        top_p=0.1,
    )
    return response.choices[0].message.content

print(gpt_4o_mini(test[0]))

In [ ]:
evaluate(gpt_4o_mini, test, size=20, workers=1)

## Gemini 2.5 Flash

In [ ]:
def gemini_flash(item):
    response = completion(
        model="openrouter/google/gemini-2.5-flash",
        messages=messages_for(item),
        temperature=0.0,
        top_p=0.1,
    )
    return response.choices[0].message.content

print(gemini_flash(test[0]))

In [ ]:
evaluate(gemini_flash, test, size=20, workers=1)

## Claude 3.5 Haiku

In [ ]:
def claude_haiku(item):
    response = completion(
        model="openrouter/anthropic/claude-3.5-haiku",
        messages=messages_for(item),
        temperature=0.0,
        top_p=0.1,
    )
    return response.choices[0].message.content

print(claude_haiku(test[0]))

In [ ]:
evaluate(claude_haiku, test, size=20, workers=1)

## Conclusion

By setting `temperature=0.0` and `top_p=0.1`, frontier models become deterministic and produce more consistent price estimates without any fine-tuning. This is a lightweight alternative when fine-tuning isn't practical or cost-effective.